In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [9]:
# --- Hyperparameters ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.0002
z_dim = 100
batch_size = 64
epochs = 50

In [10]:
# --- Data Preparation ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # Scale to [-1, 1]
])
dataset = datasets.MNIST(root="dataset/", train=True, transform=transform, download=True)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [11]:
# --- Models ---
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.disc = nn.Sequential(
            nn.Linear(784, 128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.disc(x)

class Generator(nn.Module):
    def __init__(self, z_dim):
        super().__init__()
        self.gen = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.1),
            nn.Linear(256, 784),
            nn.Tanh(), # Output must be in range [-1, 1]
        )

    def forward(self, x):
        return self.gen(x)

In [12]:
# Initialize
gen = Generator(z_dim).to(device)
disc = Discriminator().to(device)
opt_gen = optim.Adam(gen.parameters(), lr=lr)
opt_disc = optim.Adam(disc.parameters(), lr=lr)
criterion = nn.BCELoss()

In [13]:
for epoch in range(epochs):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.view(-1, 784).to(device)
        batch_curr = real.shape[0]

        ### Train Discriminator: max log(D(x)) + log(1 - D(G(z)))
        noise = torch.randn(batch_curr, z_dim).to(device)
        fake = gen(noise)

        disc_real = disc(real).view(-1)
        lossD_real = criterion(disc_real, torch.ones_like(disc_real))

        disc_fake = disc(fake).view(-1)
        lossD_fake = criterion(disc_fake, torch.zeros_like(disc_fake))

        lossD = (lossD_real + lossD_fake) / 2
        disc.zero_grad()
        lossD.backward(retain_graph=True)
        opt_disc.step()

        ### Train Generator: min log(1 - D(G(z))) <-> max log(D(G(z)))
        output = disc(fake).view(-1)
        lossG = criterion(output, torch.ones_like(output))

        gen.zero_grad()
        lossG.backward()
        opt_gen.step()

    print(f"Epoch [{epoch}/{epochs}] Loss D: {lossD:.4f}, Loss G: {lossG:.4f}")



Epoch [0/50] Loss D: 0.2881, Loss G: 1.3187
Epoch [1/50] Loss D: 0.5832, Loss G: 0.8505
Epoch [2/50] Loss D: 0.7413, Loss G: 0.7032
Epoch [3/50] Loss D: 0.4827, Loss G: 1.1910
Epoch [4/50] Loss D: 0.3446, Loss G: 1.3779
Epoch [5/50] Loss D: 0.7799, Loss G: 0.6389
Epoch [6/50] Loss D: 0.8223, Loss G: 0.7225
Epoch [7/50] Loss D: 0.7081, Loss G: 0.6906
Epoch [8/50] Loss D: 0.4075, Loss G: 1.2878
Epoch [9/50] Loss D: 0.7680, Loss G: 0.7182
Epoch [10/50] Loss D: 0.4747, Loss G: 1.0977
Epoch [11/50] Loss D: 0.3411, Loss G: 1.3506
Epoch [12/50] Loss D: 0.3253, Loss G: 1.4921
Epoch [13/50] Loss D: 0.4379, Loss G: 1.5002
Epoch [14/50] Loss D: 0.5661, Loss G: 0.9253
Epoch [15/50] Loss D: 0.6438, Loss G: 0.9717
Epoch [16/50] Loss D: 0.4771, Loss G: 1.0838
Epoch [17/50] Loss D: 0.3613, Loss G: 1.6585
Epoch [18/50] Loss D: 0.3423, Loss G: 1.5382
Epoch [19/50] Loss D: 0.4106, Loss G: 1.6191
Epoch [20/50] Loss D: 0.6450, Loss G: 1.2998
Epoch [21/50] Loss D: 0.4113, Loss G: 1.6412
Epoch [22/50] Loss D